In [1]:
import os
import pandas as pd

In [2]:
df = pd.read_json("raw_dataset.json")

In [33]:
# Import necessary modules for Qwen3 inference
from src.ai_code_reviewer.models.inference import ReviewModel
from src.ai_code_reviewer.models.config import ModelConfig, GenerationConfig

# Initialize model with Qwen3 1.7B configuration
model_config = ModelConfig(
    model_name="Qwen/Qwen3.5-2B",
    device_map="auto",
    torch_dtype="bfloat16",
    trust_remote_code=True,
    load_in_4bit=False,
    max_input_length=4096
)

# Initialize generation configuration
gen_config = GenerationConfig(
    max_new_tokens=1024,
    temperature=0.4,
    top_p=1.0,
)

# Create and load the model
review_model = ReviewModel(model_config)
review_model.load()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [34]:
TEST_PROMPT = """You are a senior Python code reviewer.

Your task is to review the changed Python file in a pull request and identify blocking issues.

A blocking issue is a problem that can break correctness, reliability, security, or expected production behavior.
Do not report style issues, formatting issues, naming preferences, documentation suggestions, missing tests, or optional refactorings.

Review rules:
- Focus on issues introduced by the patch or clearly exposed by the changed lines.
- Use the provided repository context only when it helps you understand the changed code.
- Report only issues supported by the code and context shown below.
- Prefer high precision over high recall.
- Do not report duplicate or speculative issues.
- Every issue must point to a line range in the changed file.
- Keep comments short, specific, and actionable.

Return valid JSON with exactly one top-level field: "issues".

Format:
{
  "issues": [
    {
      "line_range": {"start": <int>, "end": <int>},
      "comment": "<short review comment>"
    }
  ]
}

If there are no blocking issues, return:
{
  "issues": []
}

Return JSON only. Do not add markdown fences or extra text.

Repository metadata:
Repository: acme/data-utils (0 stars)

[requirements.txt]
pandas==2.1.0
numpy==1.26.0

Pull request metadata:
PR title: Add average price calculation to analytics module
PR description:
Added a helper to compute the average price from a list of transactions.

Changed file patch:
File: src/analytics/pricing.py
   1 | import typing as tp
   2 |
   3 |
-  4 | def average_price(transactions: tp.List[dict]) -> float:
-  5 |     total = 0
-  6 |     for t in transactions:
-  7 |         total += t["price"]
-  8 |     return total / len(transactions)
+  4 | def average_price(transactions: tp.List[dict]) -> float:
+  5 |     prices = [t["price"] for t in transactions]
+  6 |     total = sum(prices)
+  7 |     return total / len(transactions)
   8 |
   9 |
  10 | def discounted_prices(
  11 |     transactions: tp.List[dict], discount: float
  12 | ) -> tp.List[float]:
  13 |     return [t["price"] * (1 - discount) for t in transactions]

Definitions used by the changed file:
None

Code that uses definitions from the changed file:
--- src/api/routes.py ---
from src.analytics.pricing import average_price

def get_average(request):
    txns = request.json.get("transactions", [])
    return {"average": average_price(txns)}

Now, find blocking issues in the changed file patch and answer in the given format."""

In [35]:
response = review_model.generate_raw(TEST_PROMPT)
print(response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


{
  "issues": [
    {
      "line_range": {"start": 4, "end": 7},
      "comment": "Using `sum()` on a list comprehension is more efficient than accumulating in a variable and then dividing by length, especially for large transaction lists."
    }
  ]
}

